In [1]:
import random
import math
from collections import deque
import matplotlib.pyplot as plt


# -----------------------------
# Exponential Random Variable
# -----------------------------
def exp_rv(mean):
    u = random.random()
    return -mean * math.log(u)


# -----------------------------
# M/M/1 Simulation
# -----------------------------
def mm1_simulation(mean_interarrival, mean_service, max_customers, seed=1):

    random.seed(seed)

    clock = 0.0
    server_busy = False
    queue = deque()

    next_arrival = exp_rv(mean_interarrival)
    next_departure = float("inf")

    num_completed = 0
    total_delay = 0.0

    area_queue = 0.0
    area_busy = 0.0
    last_event_time = 0.0

    # For plotting
    time_points = [0]
    queue_lengths = [0]
    delays = []

    while num_completed < max_customers:

        # Determine next event
        if next_arrival < next_departure:
            event_time = next_arrival
            event_type = "arrival"
        else:
            event_time = next_departure
            event_type = "departure"

        # Update statistical areas
        time_passed = event_time - last_event_time
        area_queue += len(queue) * time_passed
        area_busy += (1 if server_busy else 0) * time_passed

        clock = event_time
        last_event_time = clock

        if event_type == "arrival":

            # Schedule next arrival
            next_arrival = clock + exp_rv(mean_interarrival)

            if not server_busy:
                server_busy = True
                next_departure = clock + exp_rv(mean_service)
            else:
                queue.append(clock)

        else:  # departure

            num_completed += 1

            if len(queue) == 0:
                server_busy = False
                next_departure = float("inf")
            else:
                arrival_time = queue.popleft()
                delay = clock - arrival_time
                total_delay += delay
                delays.append(delay)
                next_departure = clock + exp_rv(mean_service)

        # Store for graph
        time_points.append(clock)
        queue_lengths.append(len(queue))

    # Final calculations
    sim_time = clock
    avg_delay = total_delay / max_customers
    avg_queue = area_queue / sim_time
    utilization = area_busy / sim_time

    return avg_delay, avg_queue, utilization, sim_time, time_points, queue_lengths, delays


# -----------------------------
# Plot Graphs
# -----------------------------
def plot_graph(time_points, queue_lengths, delays):

    # Queue length vs time
    plt.figure()
    plt.step(time_points, queue_lengths, where="post")
    plt.xlabel("Time")
    plt.ylabel("Number in Queue")
    plt.title("M/M/1 Queue Length vs Time")
    plt.grid(True)

    # Delay Histogram
    plt.figure()
    if len(delays) > 0:
        plt.hist(delays, bins=30)
    plt.xlabel("Delay in Queue")
    plt.ylabel("Frequency")
    plt.title("Histogram of Queue Delay")
    plt.grid(True)

    plt.show()


# -----------------------------
# Main
# -----------------------------
if __name__ == "__main__":

    mean_interarrival = float(input("Enter mean inter-arrival time: "))
    mean_service = float(input("Enter mean service time: "))
    max_customers = int(input("Enter max number of customers: "))

    avg_delay, avg_queue, utilization, sim_time, time_points, queue_lengths, delays = mm1_simulation(
        mean_interarrival, mean_service, max_customers
    )

    print("\n--- Simulation Results ---")
    print("Average delay in queue (Wq):", round(avg_delay, 6))
    print("Average number in queue (Lq):", round(avg_queue, 6))
    print("Server utilization (rho):", round(utilization, 6))
    print("Simulation ended at time:", round(sim_time, 6))

    plot_graph(time_points, queue_lengths, delays)

ValueError: could not convert string to float: 'Enter '